# 02 - Exploratory Data Analysis (EDA)
**ZHAW AI-Applications: Skin Lesion Classifier**

Analyzes the HAM10000 dataset and saves all requested plots to `docs/screenshots/`.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve()))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'data' / 'processed').exists() else cwd.parent
sys.path.insert(0, str(ROOT))

PROC_DIR = ROOT / 'data' / 'processed'
RAW_DIR = ROOT / 'data' / 'raw'
FIG_DIR = ROOT / 'docs' / 'screenshots'
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {'mel': 'Melanoma', 'nv': 'Melanocytic nevi', 'bcc': 'Basal cell carcinoma',
               'akiec': 'Actinic keratoses', 'bkl': 'Benign keratosis', 'df': 'Dermatofibroma', 'vasc': 'Vascular lesions'}
CLASS_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_PALETTE = sns.color_palette('husl', 7)

train = pd.read_csv(PROC_DIR / 'train.csv')
val = pd.read_csv(PROC_DIR / 'val.csv')
test = pd.read_csv(PROC_DIR / 'test.csv')
df = pd.concat([train, val, test], ignore_index=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

print(f'EDA setup complete: {len(df):,} samples')

EDA setup complete: 10,015 samples


## 1. Class Distribution

In [ ]:
counts = df['dx'].value_counts().reindex(CLASS_ORDER)
labels = [CLASS_NAMES[k] for k in CLASS_ORDER]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(labels, counts.values, color=sns.color_palette('Set2', len(labels)))
ax.set_title('HAM10000 Class Distribution')
ax.set_ylabel('Samples')
ax.set_xlabel('Class')
ax.tick_params(axis='x', rotation=25)
ax.set_ylim(0, counts.max() * 1.15)

for bar, pct in zip(bars, (counts.values / len(df) * 100)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + counts.max() * 0.01,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR / 'class_distribution.png', dpi=150)
plt.show()

print(counts.to_string())
print(f'Class imbalance ratio (max/min): {counts.max() / counts.min():.1f}x')

## 2. Age Distribution

In [ ]:
age = df['age'].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(age, bins=20, kde=True, ax=ax, color='#2A9D8F')
ax.set_title('Age Distribution of Patients')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.set_xlim(0, 90)

fig.tight_layout()
fig.savefig(FIG_DIR / 'age_distribution.png', dpi=150)
plt.show()

print(f'Age count: {len(age):,}')
print(f'Age mean: {age.mean():.2f}')
print(f'Age median: {age.median():.1f}')
print(f'Age min/max: {age.min():.0f}/{age.max():.0f}')
print(pd.cut(age, bins=[0, 20, 40, 60, 80, 100], right=False, include_lowest=True).value_counts().sort_index().to_string())

## 3. Sex Distribution

In [ ]:
sex_order = ['female', 'male', 'unknown']
sex_counts = df['sex'].fillna('unknown').replace('', 'unknown').value_counts().reindex(sex_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(sex_counts.index, sex_counts.values, color=['#E76F51', '#264653', '#A8A8A8'])
ax.set_title('Sex Distribution of Patients')
ax.set_xlabel('Sex')
ax.set_ylabel('Count')

for bar, value in zip(bars, sex_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + max(sex_counts.values) * 0.01,
            f'{value:,}', ha='center', va='bottom', fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR / 'sex_distribution.png', dpi=150)
plt.show()

print(sex_counts.to_string())
print((sex_counts / len(df) * 100).round(2).to_string())

## 4. Localization Distribution

In [ ]:
loc_counts = df['localization'].fillna('unknown').replace('', 'unknown').value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(loc_counts.index, loc_counts.values, color='#457B9D')
ax.set_title('Body Site Distribution')
ax.set_xlabel('Count')
ax.set_ylabel('Localization')

fig.tight_layout()
fig.savefig(FIG_DIR / 'localization_distribution.png', dpi=150)
plt.show()

print(loc_counts.sort_values(ascending=False).head(10).to_string())
print((loc_counts.sort_values(ascending=False).head(10) / len(df) * 100).round(2).to_string())

## 5. Example Images per Class

## 6. Key Findings

- `nv` is the most frequent class with 6,705 / 10,015 samples (66.95%), while `df` is the rarest with 115 samples (1.15%).
- The class imbalance ratio is 58.3x between the largest and smallest classes.
- Patient age is concentrated in the 40-60 range: 4,537 samples (45.28%), with a mean age of 51.85 years and a median of 50.0 years.
- Sex is slightly male-skewed: 5,406 male cases (53.98%) versus 4,552 female cases (45.45%).
- The two most common body sites are back with 2,192 cases (21.89%) and lower extremity with 2,077 cases (20.74%), together 4,269 cases (42.63%).

In [ ]:
sample_counts = df.groupby('dx', sort=False).head(3).copy()
sample_counts['dx'] = pd.Categorical(sample_counts['dx'], categories=CLASS_ORDER, ordered=True)
sample_counts = sample_counts.sort_values(['dx', 'image_id'])

fig, axes = plt.subplots(3, 7, figsize=(21, 10))
for col, cls in enumerate(CLASS_ORDER):
    class_rows = df[df['dx'] == cls].head(3)
    for row in range(3):
        ax = axes[row, col]
        ax.axis('off')
        if row < len(class_rows):
            image_path = class_rows.iloc[row]['image_path']
            try:
                image = Image.open(image_path).convert('RGB')
                ax.imshow(image)
            except Exception as exc:
                ax.text(0.5, 0.5, 'Image\nmissing', ha='center', va='center', fontsize=10)
        if row == 0:
            ax.set_title(CLASS_NAMES[cls], fontsize=11)

fig.suptitle('Example Images per Class', fontsize=16, y=0.98)
fig.tight_layout()
fig.savefig(FIG_DIR / 'example_images_per_class.png', dpi=150)
plt.show()

print('Saved example_images_per_class.png with 3 samples for each of 7 classes')
